# 🗓️ Exam Scheduling Algorithms
This notebook provides an interactive interface for the Exam Scheduling project.  
It covers four steps: **Setup → Compile → Generate Data → Run Algorithms → View Results**.

> **Important:** Run this notebook from the **project root directory** (the folder that contains `codePortion/`).  
> In JupyterLab: *File → Open from Path* or launch with `jupyter lab` from the project root.


---
## 1 · Setup
Create all required directories and confirm the working directory is the project root.


In [ ]:
import os, sys, json, subprocess
from pathlib import Path

# ── Verify we're at the project root ──────────────────────────────────────
root = Path.cwd()
code_dir = root / "codePortion"

if not code_dir.exists():
    print("⚠️  'codePortion/' not found in the current directory.")
    print(f"   Current directory: {root}")
    print("   Please restart JupyterLab from the project root, or change")
    print("   directory below and re-run this cell.")
    print()
    # Uncomment and edit the line below if needed:
    # os.chdir("/path/to/your/project/root")
else:
    print(f"✅ Project root confirmed: {root}")

# ── Create directory structure ────────────────────────────────────────────
dirs = [
    code_dir / "Data" / "Courses",
    code_dir / "Data" / "Students",
    code_dir / "Data" / "Locations",
    code_dir / "Data" / "Schedules",
    code_dir / "alg_binaries",
]
for d in dirs:
    d.mkdir(parents=True, exist_ok=True)

print("✅ All data directories are ready.")
print()
print("Directory layout:")
for d in dirs:
    print(f"   {d.relative_to(root)}")


---
## 2 · Compile C Binaries
Compiles all four algorithm binaries from source. Re-run this cell any time you modify the C files.

**Prerequisites:** `gcc` must be installed.  
- macOS: `xcode-select --install`  
- Ubuntu/Debian: `sudo apt install build-essential`  
- Windows (WSL): `sudo apt install build-essential`


In [ ]:
src  = root / "codePortion"
bins = src / "alg_binaries"
cjson = src / "cJSON.c"

# Map: binary name → source file
algorithms = {
    "bf": src / "brute_force.c",
    "gr": src / "greedy_algorithm.c",
    "gc": src / "graph_coloring.c",
    "ga": src / "genetic_algorithm.c",
}

all_ok = True
for name, c_file in algorithms.items():
    out = bins / name
    cmd = ["gcc", "-O2", "-o", str(out), str(c_file), str(cjson), "-lm"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print(f"✅ Compiled {name}  →  {out.relative_to(root)}")
    else:
        print(f"❌ Failed to compile {name}")
        print(result.stderr)
        all_ok = False

print()
print("✅ All binaries compiled successfully." if all_ok else "⚠️  One or more compilations failed.")


---
## 3 · Generate Data

### 3a · Create a Sample Locations File
The C algorithms require a `locations.json` file. A sample one is created here if it doesn't already exist.


In [ ]:
locations_path = code_dir / "Data" / "Locations" / "locations.json"

if not locations_path.exists():
    sample_locations = {
        "locations": [
            "Building-A Room-101",
            "Building-A Room-102",
            "Building-B Room-201",
            "Building-B Room-202",
            "Building-C Auditorium"
        ]
    }
    with open(locations_path, "w") as f:
        json.dump(sample_locations, f, indent=4)
    print(f"✅ Created sample locations file: {locations_path.relative_to(root)}")
else:
    with open(locations_path) as f:
        existing = json.load(f)
    print(f"ℹ️  Locations file already exists with {len(existing['locations'])} location(s).")

print()
print("Locations available:")
with open(locations_path) as f:
    locs = json.load(f)
for loc in locs["locations"]:
    print(f"   • {loc}")


### 3b · Generate Courses
Edit the variables below, then run the cell.


In [ ]:
# ── Configure these ───────────────────────────────────────────────────────
COURSE_FILE    = "courses"   # Output filename (saved in Data/Courses/)
NUM_COURSES    = 20          # Number of courses to generate

# ── Add the codePortion directory to sys.path so we can import directly ──
if str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))

from generate_data import generateNewCourses

output_path = code_dir / "Data" / "Courses" / (COURSE_FILE + ".json")
if output_path.exists():
    print(f"ℹ️  '{COURSE_FILE}.json' already exists. Delete it first to regenerate.")
else:
    # generateNewCourses uses its own __file__-relative path resolution
    os.chdir(code_dir)
    generateNewCourses(COURSE_FILE, NUM_COURSES)
    os.chdir(root)
    print(f"✅ Generated {NUM_COURSES} courses → {output_path.relative_to(root)}")

# Preview
with open(output_path) as f:
    course_data = json.load(f)

total = sum(len(d["courses"]) for d in course_data["departments"])
print(f"\nTotal courses across all departments: {total}")
for dept in course_data["departments"]:
    print(f"   {dept['name']}: {len(dept['courses'])} course(s)")


### 3c · Generate Students


In [ ]:
# ── Configure these ───────────────────────────────────────────────────────
EXISTING_COURSES_FILE = "courses"   # Must match the file generated above
STUDENT_FILE          = "students"  # Output filename (saved in Data/Students/)
NUM_STUDENTS          = 50          # Number of students to generate

from generate_data import generateNewStudents

output_path = code_dir / "Data" / "Students" / (STUDENT_FILE + ".json")
if output_path.exists():
    print(f"ℹ️  '{STUDENT_FILE}.json' already exists. Delete it first to regenerate.")
else:
    os.chdir(code_dir)
    generateNewStudents(EXISTING_COURSES_FILE + ".json", STUDENT_FILE, NUM_STUDENTS)
    os.chdir(root)
    print(f"✅ Generated {NUM_STUDENTS} students → {output_path.relative_to(root)}")

# Preview
with open(output_path) as f:
    student_data = json.load(f)

from collections import Counter
majors = Counter(s["major"] for s in student_data["students"])
print(f"\nTotal students: {len(student_data['students'])}")
print("Breakdown by major:")
for major, count in sorted(majors.items()):
    print(f"   {major}: {count}")


---
## 4 · Run Algorithms

The helper function below handles launching a compiled binary, piping the  
required file-name inputs via stdin, and capturing all output for display.

> **File name inputs:** All four algorithms ask for just the **filename** (not the full path).  
> The binaries resolve paths relative to `./codePortion/Data/<subfolder>/` automatically.


In [ ]:
def run_algorithm(binary_name: str,
                  students_file: str,
                  courses_file: str,
                  locations_file: str,
                  output_file: str,
                  extra_stdin: str = "") -> dict:
    """
    Run a compiled scheduling binary and return a results dict.

    Parameters
    ----------
    binary_name    : one of 'bf', 'gr', 'gc', 'ga'
    students_file  : filename only, e.g. 'students.json'
    courses_file   : filename only, e.g. 'courses.json'
    locations_file : filename only, e.g. 'locations.json'
    output_file    : filename only, e.g. 'schedule_bf.json'
    extra_stdin    : additional stdin lines (used by genetic algorithm)
    """
    binary = root / "codePortion" / "alg_binaries" / binary_name
    if not binary.exists():
        raise FileNotFoundError(f"Binary not found: {binary}. Did you run Section 2?")

    # Each binary reads: students_file, courses_file, locations_file, output_file
    stdin_text = "\n".join([
        students_file,
        courses_file,
        locations_file,
        output_file,
        extra_stdin,
        ""  # trailing newline
    ])

    result = subprocess.run(
        [str(binary)],
        input=stdin_text,
        capture_output=True,
        text=True,
        cwd=str(root)   # binaries resolve paths from project root
    )

    return {
        "returncode": result.returncode,
        "stdout": result.stdout,
        "stderr": result.stderr,
        "output_file": root / "codePortion" / "Data" / "Schedules" / output_file,
    }


def print_result(res: dict):
    print("─" * 60)
    print(res["stdout"])
    if res["stderr"]:
        print("STDERR:", res["stderr"])
    if res["returncode"] != 0:
        print(f"⚠️  Process exited with code {res['returncode']}")
    else:
        print(f"✅ Output written to: {res['output_file'].relative_to(root)}")
    print("─" * 60)


print("✅ Helper functions defined. Ready to run algorithms.")


### 4a · Brute Force
Explores all valid slot assignments via backtracking with incremental conflict pruning.  
Best for small datasets — runtime grows exponentially with the number of courses.


In [ ]:
# ── Configure these ───────────────────────────────────────────────────────
BF_STUDENTS  = "students.json"
BF_COURSES   = "courses.json"
BF_LOCATIONS = "locations.json"
BF_OUTPUT    = "schedule_bf.json"

print("Running Brute Force... (may take a while for large datasets)")
res_bf = run_algorithm("bf", BF_STUDENTS, BF_COURSES, BF_LOCATIONS, BF_OUTPUT)
print_result(res_bf)


### 4b · Greedy Algorithm
Assigns each exam to the earliest conflict-free slot in one pass.  
Very fast; produces a valid schedule when one exists but makes no optimality guarantees.


In [ ]:
# ── Configure these ───────────────────────────────────────────────────────
GR_STUDENTS  = "students.json"
GR_COURSES   = "courses.json"
GR_LOCATIONS = "locations.json"
GR_OUTPUT    = "schedule_gr.json"

print("Running Greedy Algorithm...")
res_gr = run_algorithm("gr", GR_STUDENTS, GR_COURSES, GR_LOCATIONS, GR_OUTPUT)
print_result(res_gr)


### 4c · Graph Coloring
Models courses as graph nodes; shared-student edges are coloured so no two  
adjacent nodes share a time slot. Produces near-optimal schedules efficiently.


In [ ]:
# ── Configure these ───────────────────────────────────────────────────────
GC_STUDENTS  = "students.json"
GC_COURSES   = "courses.json"
GC_LOCATIONS = "locations.json"
GC_OUTPUT    = "schedule_gc.json"

print("Running Graph Coloring Algorithm...")
res_gc = run_algorithm("gc", GC_STUDENTS, GC_COURSES, GC_LOCATIONS, GC_OUTPUT)
print_result(res_gc)


### 4d · Genetic Algorithm
Evolves a population of schedules over many generations using selection, crossover,  
and mutation. Supports optional parameter tuning — see `GA_CUSTOM_PARAMS` below.


In [ ]:
# ── Configure these ───────────────────────────────────────────────────────
GA_STUDENTS  = "students.json"
GA_COURSES   = "courses.json"
GA_LOCATIONS = "locations.json"
GA_OUTPUT    = "schedule_ga.json"

# Set to True to supply custom parameters, or False to use algorithm defaults
GA_CUSTOM_PARAMS = False

# Only used when GA_CUSTOM_PARAMS = True
GA_POPULATION_SIZE  = 100
GA_NUM_GENERATIONS  = 500
GA_MUTATION_RATE    = 0.10
GA_ELITISM_COUNT    = 5
GA_TOURNAMENT_SIZE  = 5

# Build the extra stdin for the genetic algorithm's parameter prompt
if GA_CUSTOM_PARAMS:
    extra = "\n".join([
        "Y",
        str(GA_POPULATION_SIZE),
        str(GA_NUM_GENERATIONS),
        str(GA_MUTATION_RATE),
        str(GA_ELITISM_COUNT),
        str(GA_TOURNAMENT_SIZE),
    ])
else:
    extra = "N"

print("Running Genetic Algorithm...")
res_ga = run_algorithm("ga", GA_STUDENTS, GA_COURSES, GA_LOCATIONS, GA_OUTPUT, extra_stdin=extra)
print_result(res_ga)


---
## 5 · View & Compare Results

Load one or more output schedules and display them as formatted tables.


In [ ]:
def load_schedule(filename: str) -> dict | None:
    """Load a schedule JSON from Data/Schedules/. Returns None if not found."""
    path = root / "codePortion" / "Data" / "Schedules" / filename
    if not path.exists():
        print(f"⚠️  {filename} not found. Run the corresponding algorithm first.")
        return None
    with open(path) as f:
        return json.load(f)


def display_schedule(data: dict, title: str):
    """Pretty-print a schedule as a sorted table."""
    if data is None:
        return

    print(f"\n{'═'*60}")
    print(f"  {title}")
    print(f"{'═'*60}")

    # Sort by day then time
    day_order  = {"M": 0, "T": 1, "W": 2, "R": 3, "F": 4}
    time_order = {
        "8:00am-10:00am":    0,
        "10:30am-12:30pm":   1,
        "1:00pm-3:00pm":     2,
        "3:30pm-5:30pm":     3,
    }

    rows = []
    for course_id, slot in data.items():
        if isinstance(slot, dict):
            rows.append((course_id, slot.get("day","?"), slot.get("time","?"), slot.get("location","?")))

    rows.sort(key=lambda r: (day_order.get(r[1], 99), time_order.get(r[2], 99)))

    print(f"  {'Day':<5} {'Time':<22} {'Location':<30} Course ID")
    print(f"  {'─'*4} {'─'*21} {'─'*29} {'─'*36}")
    for course_id, day, time, location in rows:
        print(f"  {day:<5} {time:<22} {location:<30} {course_id}")

    print(f"\n  Total exams scheduled: {len(rows)}")


print("✅ Display helpers loaded.")


In [ ]:
# ── View individual schedules ─────────────────────────────────────────────
display_schedule(load_schedule("schedule_bf.json"), "Brute Force Schedule")
display_schedule(load_schedule("schedule_gr.json"), "Greedy Schedule")
display_schedule(load_schedule("schedule_gc.json"), "Graph Coloring Schedule")
display_schedule(load_schedule("schedule_ga.json"), "Genetic Algorithm Schedule")


### 5b · Time-Slot Utilisation Comparison
How many distinct time slots does each algorithm use?


In [ ]:
schedule_files = {
    "Brute Force":       "schedule_bf.json",
    "Greedy":            "schedule_gr.json",
    "Graph Coloring":    "schedule_gc.json",
    "Genetic Algorithm": "schedule_ga.json",
}

print(f"{'Algorithm':<22} {'Exams Scheduled':<18} {'Unique Time Slots Used'}")
print(f"{'─'*21} {'─'*17} {'─'*22}")

for alg_name, fname in schedule_files.items():
    data = load_schedule(fname)
    if data is None:
        print(f"{alg_name:<22} {'—':<18} —")
        continue
    slots_used = set()
    exams = 0
    for course_id, slot in data.items():
        if isinstance(slot, dict):
            exams += 1
            slots_used.add((slot.get("day"), slot.get("time")))
    print(f"{alg_name:<22} {exams:<18} {len(slots_used)}")


### 5c · Conflict Check
Verify that no student has two exams at the same time in a given schedule.


In [ ]:
def check_conflicts(schedule_file: str, students_file: str = "students.json") -> None:
    """
    Load a schedule and a student list, then report any time-slot conflicts.
    A conflict = one student has two or more exams in the same (day, time) slot.
    """
    schedule = load_schedule(schedule_file)
    if schedule is None:
        return

    student_path = root / "codePortion" / "Data" / "Students" / students_file
    if not student_path.exists():
        print(f"⚠️  {students_file} not found.")
        return

    with open(student_path) as f:
        student_data = json.load(f)

    conflicts = 0
    for student in student_data["students"]:
        seen_slots: dict = {}
        for course_id in student["enrolledCourses"]:
            slot = schedule.get(course_id)
            if slot and isinstance(slot, dict):
                key = (slot.get("day"), slot.get("time"))
                if key in seen_slots:
                    conflicts += 1
                    break   # one conflict per student is enough to flag
                seen_slots[key] = course_id

    print(f"Schedule : {schedule_file}")
    print(f"Students : {students_file}")
    if conflicts == 0:
        print("✅ No conflicts detected — every student has at most one exam per time slot.")
    else:
        print(f"⚠️  {conflicts} student(s) have conflicting exam time slots.")
    print()


check_conflicts("schedule_bf.json")
check_conflicts("schedule_gr.json")
check_conflicts("schedule_gc.json")
check_conflicts("schedule_ga.json")


---
## 6 · Clean Up (Optional)
Delete generated data files so you can start fresh.  
Uncomment only what you want to delete, then run the cell.


In [ ]:
import shutil

# ── Uncomment the lines for what you want to delete ───────────────────────

# Delete all generated courses
# shutil.rmtree(code_dir / "Data" / "Courses", ignore_errors=True)
# (code_dir / "Data" / "Courses").mkdir(exist_ok=True)
# print("🗑️  Courses deleted.")

# Delete all generated students
# shutil.rmtree(code_dir / "Data" / "Students", ignore_errors=True)
# (code_dir / "Data" / "Students").mkdir(exist_ok=True)
# print("🗑️  Students deleted.")

# Delete all generated schedules
# shutil.rmtree(code_dir / "Data" / "Schedules", ignore_errors=True)
# (code_dir / "Data" / "Schedules").mkdir(exist_ok=True)
# print("🗑️  Schedules deleted.")

# Delete compiled binaries
# shutil.rmtree(code_dir / "alg_binaries", ignore_errors=True)
# (code_dir / "alg_binaries").mkdir(exist_ok=True)
# print("🗑️  Binaries deleted.")

print("Nothing deleted. Uncomment a block above to clean up.")
